<a href="https://colab.research.google.com/github/opherdonchin/BayesShortCourse/blob/main/sleep/09_exgaussian_distributional.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sleep deprivation 9 — Ex-Gaussian distributional model

Notebooks 6–8 represented the right skew of reaction times with a lognormal likelihood, which models log reaction time. Their mean structure therefore acted multiplicatively, and Notebook 8 gave each participant their own residual scale on that log scale.

This notebook represents right skew differently. An **ex-Gaussian** likelihood adds an exponential tail to a Gaussian distribution, so the skew comes from the shape of the likelihood rather than from a log transform. The mean structure returns to Notebook 5's, in milliseconds, and each participant keeps their own residual scale, built as in Notebook 8. The new pieces are the likelihood and its tail, whose priors are chosen in milliseconds and then translated to the log scale.

## Setup

This course pins PyMC and the modular ArviZ packages for reproducibility because their APIs can change across major versions.

In [ ]:
%pip install -q \
    "pandas==2.2.3" \
    "pymc==6.3.2" \
    "arviz-base==1.3.0" \
    "arviz-stats==1.3.2" \
    "arviz-plots[matplotlib]==1.3.1"

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xarray as xr
import pymc as pm
import arviz_base as azb
import arviz_plots as azp
import arviz_stats as azs

RANDOM_SEED = 20260924
azp.style.use("arviz-variat")

print("PyMC:", pm.__version__)
print("arviz-base:", azb.__version__)
print("arviz-plots:", azp.__version__)
print("arviz-stats:", azs.__version__)

## Data

The original study contains two adaptation/training days followed by a baseline measurement and then seven nights of severe sleep restriction. You can read about the original study here: [Belenky J Sleep Res. 2003](https://doi.org/10.1046/j.1365-2869.2003.00337.x)

Following the chapter, we drop original days 0–1 and subtract 2 from the remaining day number. Therefore **`Days = 0` is the baseline measurement before sleep deprivation begins**.

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/vincentarelbundock/Rdatasets/master/csv/lme4/sleepstudy.csv"

sleep = pd.read_csv(DATA_URL).drop(columns="rownames")
sleep = sleep.loc[sleep["Days"] >= 2].copy()
sleep["Days"] = sleep["Days"] - 2
sleep["Subject"] = sleep["Subject"].astype(str)
sleep = sleep.reset_index(drop=True)

participants = sorted(sleep["Subject"].unique(), key=int)
participant_to_idx = {participant: i for i, participant in enumerate(participants)}
participant_idx = sleep["Subject"].map(participant_to_idx).to_numpy()

assert participant_idx.min() == 0
assert participant_idx.max() == len(participants) - 1
assert np.array_equal(
    np.asarray(participants)[participant_idx],
    sleep["Subject"].to_numpy(),
)

print(f"{len(participants)} participants, {len(sleep)} observations")
print(f"Days: {sleep['Days'].min()} to {sleep['Days'].max()}")
sleep.head()

### Plotting helper

The participant plotting helper from the previous notebooks is supplied. It can optionally restrict plots to selected participants using the standard ArviZ `coords` argument. Panels appear in participant order (308, 309, …, 372), left to right and top to bottom.

In [ ]:
LM_VISUALS = {
    "pe_line": {"color": "C1"},
    "ci_band": {"color": "C0"},
    "observed_scatter": {"color": "black", "alpha": 1, "zorder": 3, "s": 10},
}

PARTICIPANT_DAY = xr.Coordinates.from_pandas_multiindex(
    pd.MultiIndex.from_frame(
        sleep[["Subject", "Days"]],
        names=["participant", "day"],
    ),
    "obs_id",
)

def plot_participants(dt, group, var, coords=None):
    """One panel per participant, optionally restricted with ArviZ coords."""
    def reshape(ds):
        return ds.assign_coords(PARTICIPANT_DAY).unstack("obs_id")

    panels = xr.DataTree.from_dict({
        group: reshape(dt[group].to_dataset()),
        "observed_data": reshape(dt["observed_data"].to_dataset()),
        "constant_data": reshape(dt["constant_data"].to_dataset()),
    })

    pc = azp.plot_lm(
        panels,
        x="days",
        y=var,
        y_obs="y",
        group=group,
        plot_dim="day",
        coords=coords,
        ci_prob=(0.50, 0.90),
        ci_kind="hdi",
        point_estimate="mean",
        smooth=False,
        cols=["participant"],
        col_wrap=6,
        figure_kwargs={"figsize": (11, 5.5), "sharex": True, "sharey": True},
        visuals={**LM_VISUALS, "xlabel": False, "ylabel": False},
    )

    pc.add_legend("prob", title="HDI")
    fig = pc.get_viz("figure")
    fig.supxlabel("Days of sleep deprivation")
    fig.supylabel("Reaction time (ms)")
    return pc

## 1. Build the ex-Gaussian model

### 1.1 The model

$$
y_i \sim \operatorname{ExGaussian}(\mu_{y,i}, sd_{y,s[i]}, \nu)
$$

$$
\mu_{y,i}
=
b_{0,s[i]}
+
b_{1,s[i]}\,\mathrm{days}_i
$$

$$
b_{0,s}
\sim
\operatorname{Normal}(\mu_{b0}, sd_{b0})
$$

$$
b_{1,s}
\sim
\operatorname{Normal}(\mu_{b1}, sd_{b1})
$$

$$
\log sd_{y,s}
\sim
\operatorname{Normal}(\mu_{\log sd_y}, sd_{\log sd_y}).
$$

Here $s[i]$ identifies the participant who produced observation $i$. The second, third, and fourth lines are Notebook 5's mean structure, in milliseconds, and the last line is Notebook 8's residual-scale hierarchy.

The first line is new. An ex-Gaussian reaction time is the sum of two independent parts,

$$
y_i = g_i + e_i,
$$

where the Gaussian part $g_i \sim \operatorname{Normal}(\mu_{y,i}, sd_{y,s[i]})$ scatters symmetrically around $\mu_{y,i}$, and the exponential part $e_i \sim \operatorname{Exponential}(\mathrm{scale}=\nu)$ is never negative and has mean $\nu$. PyMC's `pm.ExGaussian(mu, sigma, nu)` uses exactly this parameterization. The tail mean $\nu$ has no participant subscript: all participants share one tail.

### 1.2 Compare ex-Gaussian distributions with different tails.

The supplied code draws three ex-Gaussian distributions with the same Gaussian part, $\mu = 250$ ms and $sd = 30$ ms, and different tail means $\nu$. The dashed lines mark each distribution's mean.

In [ ]:
rt = np.linspace(100, 800, 701)

fig, ax = plt.subplots(figsize=(7, 3.5))
for nu_value in [5.0, 50.0, 150.0]:
    # Floats, not integers: PyTensor stores 30 as an 8-bit integer, so 30**2 would overflow.
    exgaussian = pm.ExGaussian.dist(mu=250.0, sigma=30.0, nu=nu_value)
    density = np.exp(pm.logp(exgaussian, rt).eval())
    (line,) = ax.plot(rt, density, label=f"{nu_value:.0f} ms")
    ax.axvline(250.0 + nu_value, color=line.get_color(), linestyle="--", linewidth=1)

ax.set_xlabel("Reaction time (ms)")
ax.set_ylabel("Density")
ax.legend(title="Tail mean ν")
plt.show()

### 1.3 What does $\nu$ represent?

How does the distribution change as $\nu$ grows?

- answer here

### 1.4 Why is the expected reaction time $\mu_{y,i} + \nu$ rather than $\mu_{y,i}$?

Use the two parts of $y_i$ from Question 1.1, and check your answer against the dashed lines in Question 1.2.

- answer here

### 1.5 Can the ex-Gaussian produce a negative reaction time?

Compare with the lognormal likelihood of Notebooks 6–8.

- answer here

### 1.6 Why does $\mu_{y,i}$ need no log link here, unlike in Notebooks 6–8?

In Notebooks 6–8, $b_0$ and $b_1$ were on the log scale: the lognormal's location is the log of the median reaction time, so the mean structure reached milliseconds only through exponentiation. Modeling a quantity on the log scale and exponentiating it in this way is called a **log link**; Notebook 8 used one for `sd_y`. What scale are `b0` and `b1` on now, and which parameters of this model do need a log link?

- answer here

### 1.7 The supplied mean structure

The data, coordinates, and mean structure are Notebook 5's, with the same prior constants, and are supplied. As in Notebook 8, the model is created without the remaining nodes, which are added in separate `with model:` blocks.

In [ ]:
coords = {
    "obs_id": np.arange(len(sleep)),
    "participant": participants,
}

# Hyperprior constants for the intercept hierarchy (ms), from Notebook 5
mu_mu_b0 = 250
sd_mu_b0 = 100
sd_sd_b0 = 25

# Hyperprior constants for the slope hierarchy (ms/day), from Notebook 5
mu_mu_b1 = 0
sd_mu_b1 = 20
sd_sd_b1 = 10

with pm.Model(coords=coords) as model:
    days = pm.Data("days", sleep["Days"].to_numpy(), dims="obs_id")
    pidx = pm.Data("participant_idx", participant_idx, dims="obs_id")

    # Varying intercepts
    mu_b0 = pm.Normal("mu_b0", mu=mu_mu_b0, sigma=sd_mu_b0)
    sd_b0 = pm.Exponential("sd_b0", scale=sd_sd_b0)
    b0 = pm.Normal("b0", mu=mu_b0, sigma=sd_b0, dims="participant")

    # Varying slopes
    mu_b1 = pm.Normal("mu_b1", mu=mu_mu_b1, sigma=sd_mu_b1)
    sd_b1 = pm.Exponential("sd_b1", scale=sd_sd_b1)
    b1 = pm.Normal("b1", mu=mu_b1, sigma=sd_b1, dims="participant")

    # Location of the Gaussian part (ms)
    mu_y = pm.Deterministic(
        "mu_y",
        b0[pidx] + b1[pidx] * days,
        dims="obs_id",
    )

### 1.8 What do `b0` and `b1` describe in this model?

They keep Notebook 5's priors, which were chosen for the baseline reaction time and its daily change. Is `mu_b0` still the typical baseline reaction time, and is `b1` still the daily change in expected reaction time?

- answer here

### 1.9 The supplied residual-scale hierarchy

The standard deviation of the Gaussian part gets Notebook 8's participant-level hierarchy, with the same names, and is supplied. Its prior is chosen in milliseconds: we expect a typical participant's Gaussian standard deviation to be around 30 ms, with about 95% prior probability between 11 and 80 ms. On the log scale, $\log 30 \approx 3.4$ and the limits are about $3.4 \pm 2(0.5)$, so

$$
\mu_{\log sd_y} \sim \operatorname{Normal}(\log 30, 0.5).
$$

The prior for between-participant differences is Notebook 8's, $sd_{\log sd_y} \sim \operatorname{Exponential}(\mathrm{scale}=1/3)$.

In [ ]:
# Hyperprior constants for the residual-scale hierarchy (log of the Gaussian SD in ms)
mu_mu_log_sd_y = np.log(30)
sd_mu_log_sd_y = 0.5
sd_sd_log_sd_y = 1 / 3

with model:
    mu_log_sd_y = pm.Normal("mu_log_sd_y", mu=mu_mu_log_sd_y, sigma=sd_mu_log_sd_y)
    sd_log_sd_y = pm.Exponential("sd_log_sd_y", scale=sd_sd_log_sd_y)
    log_sd_y = pm.Normal(
        "log_sd_y",
        mu=mu_log_sd_y,
        sigma=sd_log_sd_y,
        dims="participant",
    )
    sd_y = pm.Deterministic("sd_y", pm.math.exp(log_sd_y), dims="participant")

### 1.10 How does this hierarchy differ from Notebook 8's?

The names and structure are Notebook 8's. What does `sd_y` measure here? Why is the prior center for `mu_log_sd_y` about 3.4 here but −2.5 in Notebook 8, while the prior for `sd_log_sd_y` is unchanged?

- answer here

### 1.11 Why does $\nu$ have no participant subscript?

Each participant has their own Gaussian standard deviation, but all share one tail. Why is that a reasonable choice for these data, and what would change if $\nu$ also varied by participant?

- answer here

### 1.12 What prior should we use for the tail mean?

Suppose we expect the tail mean $\nu$ to be around 50 ms and regard values between about 11 and 220 ms as plausible. Like `sd_y`, $\nu$ must be positive, so we give it a log link: a Normal prior for $\log \nu$. What Normal distribution puts about 95% of its probability between those limits?

- answer here

### 1.13 Add the tail to the model.

Store the constants from Question 1.12 in `mu_log_nu` and `sd_log_nu`, writing the center as `np.log(50)` so that the value chosen in milliseconds stays visible. Then, in a `with model:` block, add `log_nu` and the tail mean itself as a `pm.Deterministic` named `nu`.

In [ ]:
# answer here

### 1.14 Complete the model.

Add the expected reaction time in milliseconds (Question 1.4) as a `pm.Deterministic` named `mean_rt`, and the ex-Gaussian likelihood `y` for the observed reaction times. Each observation uses its participant's Gaussian standard deviation and the shared tail.

In [ ]:
# answer here

### 1.15 Inspect the completed model.

In [ ]:
pm.model_to_graphviz(model)

### 1.16 Which nodes are shared by all participants, and where does the tail enter the model?

- answer here

## 2. Check the prior implications

### 2.1 Criteria

The prior predictions should meet the criteria established in Notebook 1: predicted reaction times should not routinely be physically impossible, reaction times near baseline should mostly occupy a broadly plausible range, and the model should allow substantial change across the seven days without routinely generating absurd trajectories. The mean-structure priors are Notebook 5's, whose prior predictive check found them broad, with 90% bands reaching below zero late in the week (Notebook 5, Question 2.5).

Two criteria concern the new likelihood. As in Notebook 8, the residual-scale hierarchy should allow participants to differ noticeably in day-to-day variability without making enormous differences routine. The tail prior should allow anything from an almost symmetric distribution to a clearly right-skewed one (Question 1.2), without making tails of several hundred milliseconds routine.

### 2.2 Draw from the prior.

In [ ]:
with model:
    prior = pm.sample_prior_predictive(
        draws=500,
        var_names=["mu_log_sd_y", "sd_log_sd_y", "nu", "y"],
        random_seed=RANDOM_SEED,
    )

### 2.3 Plot the priors of the residual-scale hyperparameters and the tail mean.

Use `azp.plot_dist` for `mu_log_sd_y`, `sd_log_sd_y`, and `nu`, with 90% HDIs and the mean as the point estimate.

In [ ]:
# answer here

### 2.4 Are these priors reasonable?

Translate the 90% HDI for `mu_log_sd_y` into a typical Gaussian standard deviation in milliseconds, and read the tail mean `nu` directly in milliseconds. Why is the mean of the `nu` prior above 50 ms?

- answer here

### 2.5 Plot the prior predictive reaction times.

In [ ]:
plot_participants(prior, "prior_predictive", "y")
plt.show()

### 2.6 Do these prior predictions meet the criteria established in Notebook 1?

Judge support, baseline scale, and changes across days, and compare with the prior predictive checks of Notebook 5 and of Notebooks 6–8.

- answer here

### 2.7 Why does one panel's mean line zigzag?

In participant 333's panel (second row, first panel), the mean line swings from day to day, from about −170 ms to more than 700 ms, while every other panel's mean line stays near 300 ms. Which part of the model can move a single day's mean down as well as up, and which prior must be responsible?

- answer here

## 3. Fit and diagnose the model

### 3.1 Sample from the posterior.

This model is sampled with `target_accept=0.99` instead of the default 0.8. Question 3.2 asks why.

In [ ]:
with model:
    idata = pm.sample(
        draws=1000,
        tune=1500,
        chains=4,
        target_accept=0.99,
        random_seed=RANDOM_SEED,
    )

### 3.2 Why does this model need such a high `target_accept`?

With the default `target_accept` of 0.8, the sampler reports more than a dozen divergences for this model, and 0.95 usually still leaves a few. As in Notebook 8, the sampler chooses a single step size for the whole posterior during tuning, and a divergence signals that somewhere this step is too large for how sharply the posterior curves there. Here the divergences occur mostly where a steady participant, such as 309, has a very small Gaussian standard deviation. Imagine the Gaussian part of the distributions in Question 1.2 becoming very narrow while the tail keeps its mean. What happens to the left edge of the distribution, and what does that imply for how far that participant's `mu_y` can move?

- answer here

### 3.3 Check population-level diagnostics.

In [ ]:
print("Divergences:", int(idata["sample_stats"]["diverging"].sum().item()))

azs.summary(
    idata,
    var_names=["mu_b0", "sd_b0", "mu_b1", "sd_b1", "mu_log_sd_y", "sd_log_sd_y", "nu"],
    ci_prob=0.90,
    ci_kind="hdi",
    round_to=2,
)

In [ ]:
azp.plot_trace_dist(
    idata,
    var_names=["mu_b0", "sd_b0", "mu_b1", "sd_b1", "mu_log_sd_y", "sd_log_sd_y", "nu"],
);

### 3.4 Do the population-level parameters meet the diagnostic criteria?

Use the criteria established in Notebook 1: no divergences, R-hat close to 1, adequate bulk and tail ESS, and well-mixed traces.

- answer here

### 3.5 Screen all participants, then inspect a representative subset.

In [ ]:
participant_diagnostics = azs.summary(
    idata,
    var_names=["b0", "b1", "log_sd_y"],
    kind="diagnostics",
    round_to=2,
)

display(pd.DataFrame(
    {
        "value": [
            participant_diagnostics["r_hat"].max(),
            participant_diagnostics["ess_bulk"].min(),
            participant_diagnostics["ess_tail"].min(),
        ]
    },
    index=["largest R-hat", "smallest bulk ESS", "smallest tail ESS"],
))

diagnostic_participants = [
    participants[0],
    participants[len(participants) // 2],
    participants[-1],
]
diagnostic_coords = {"participant": diagnostic_participants}

azs.summary(
    idata,
    var_names=["b0", "b1", "log_sd_y"],
    coords=diagnostic_coords,
    ci_prob=0.90,
    ci_kind="hdi",
    round_to=2,
)

In [ ]:
azp.plot_trace_dist(
    idata,
    var_names=["b0", "b1", "log_sd_y"],
    coords=diagnostic_coords,
);

### 3.6 Do the participant-level parameters meet the same criteria?

- answer here

## 4. Examine the fitted ex-Gaussian components

### 4.1 Plot the posterior distribution of the population-average daily effect.

In [ ]:
# answer here

### 4.2 What range of population-average daily effects is credible?

Read the 90% HDI from your plot, and compare it with Notebook 5's estimate. In what units is it, and why?

- answer here

### 4.3 Plot the posterior distribution of the tail mean `nu`.

In [ ]:
# answer here

### 4.4 How long is the fitted tail, and how far did the data move it from its prior?

Compare your plot with the prior for `nu` in Question 2.3.

- answer here

### 4.5 What does such a short tail mean for the shape of the reaction-time distribution?

A typical participant's Gaussian standard deviation is about $e^{2.8} \approx 16$ ms (summary in Question 3.3). Which curve in Question 1.2 does a typical participant's fitted distribution resemble? Why might the tail be so much shorter than we expected?

- answer here

### 4.6 Plot each participant's Gaussian standard deviation.

Use `azp.plot_forest` for `sd_y`, with 50% and 90% HDIs.

In [ ]:
# answer here

### 4.7 How much do participants differ in day-to-day variability?

Base the answer on the participant intervals and on the posterior for `sd_log_sd_y` in the summary of Question 3.3, and express the difference between the steadiest and the most variable participants in milliseconds.

- answer here

## 5. Predictive consequences

### 5.1 Plot each participant's posterior expected reaction time.

Use `plot_participants` with the variable that gives the expected reaction time in milliseconds.

In [ ]:
# answer here

### 5.2 What uncertainty do these bands represent, and how would a plot of `mu_y` differ?

- answer here

### 5.3 Generate and plot posterior predictive reaction times.

Generate replicated values of `y` from the fitted model, add them to `idata`, and compare them with the observations using `plot_participants`.

In [ ]:
# answer here

### 5.4 What is added when we move from `mean_rt` to posterior predictive `y`?

How do the widths of a participant's bands change across the week, and how does that differ from Notebook 8's lognormal model?

- answer here

### 5.5 Does the model reproduce the participant-level data?

Apply the posterior predictive criteria from Notebook 1: participants' overall levels, changes across days, and residual variation, emphasizing discrepancies that persist across a participant's observations. Compare with Notebook 5's check, in which all participants shared one residual standard deviation.

- answer here

### 5.6 What would indicate adequate fit in an ECDF check?

The participant panels check conditional trajectories. Pooling all observations asks a different question: does the model reproduce the overall distribution of reaction times, including its right tail?

As in Notebooks 5–8, the observed ECDF should be broadly consistent with the posterior-predictive ECDFs across the whole distribution, without a persistent systematic displacement, particularly in the tails.

### 5.7 Plot the observed and posterior-predictive ECDFs.

Use `azp.plot_ppc_dist` with `kind="ecdf"`.

In [ ]:
# answer here

### 5.8 Does the model reproduce the overall distribution, including its right tail?

The fitted tail is short (Question 4.4). Where does the right skew of the pooled reaction times come from in this model?

- answer here

## 6. Summary

### 6.1 What has this notebook shown?

Summarize what the ex-Gaussian model showed about the likelihood, the priors, sampling, the fitted components, and the predictive checks.

- answer here